DH演習2026 · 教材 第53版 · 2026-r53 · 2026-11-25

<details><summary>教材の版と相談用情報</summary>

相談時は次の情報を伝えてください。

```text
DH演習2026 · 教材 第53版
教材ID: 2026-r53
教材日付: 2026-11-25
活動: text_counts
環境: Colab / Jupyter
ソース: 3bb1cdc74699c6df82735f72bff2ce617a0b4964
```

</details>

[同じ版の教材・変更点](https://nlp.lang.osaka-u.ac.jp/dh/practicum/editions.html)


# 第7回：戯曲全体から発言と人物への言及を数える

発言者の表示名と作品内IDで、同じ発言を数え比べます。

全員が1〜3節と確認問題に取り組みます。授業の教材で②を選んだ人は5節、③を選んだ人は4節へ進んでください。「授業外・任意」と付いた課題は、取り組むかどうかを自分で決めてください。

セルはShift+Enterで実行します。値を変えたら、そのセルをもう一度実行してください。

資料は芥川龍之介『二人小町』全文（[TEI-EAJ aozora_tei](https://github.com/TEI-EAJ/aozora_tei/blob/77a675fc2771936f9544505d922d4cd45075338c/data/complete/tei_lib_lv3/86_tei.xml)、CC0）です。5節の人物のタグは教材で加えた注釈です。

## 1. 戯曲全体を取得する

ノートブックは、版を固定したTEIファイルを取得します。`div`は場、`sp`は一つの発言です。

In [ ]:
#@title 操作：本文を取得する
import re
import xml.etree.ElementTree as ET

import pandas as pd
import requests

url = "https://raw.githubusercontent.com/TEI-EAJ/aozora_tei/77a675fc2771936f9544505d922d4cd45075338c/data/complete/tei_lib_lv3/86_tei.xml"
T = "{http://www.tei-c.org/ns/1.0}"
tei = ET.fromstring(requests.get(url).content)
body = tei.findall(f".//{T}body")[0]
divs = body.findall(T + "div")
print("場の数：", len(divs))
print("発言の数：", sum(len(div.findall(T + "sp")) for div in divs))

## 2. 発言を表にする

次のセルはそのまま実行してください。ふりがなの読みと括弧を除き、一つの発言を一行にします。

表の最初の行で、表示名（`speaker`）と作品内ID（`who`）を見比べてください。授業の教材の`#tama`は、原資料の`#玉造の小町`に当たります。

In [ ]:
#@title 準備：発言の表を作る
def plain(e):
    if e.tag == T + "span" and e.get("type") in ("rt", "rp"):
        return e.tail or ""
    return (e.text or "") + "".join(plain(child) for child in e) + (e.tail or "")

rows = []
for div in divs:
    for n, sp in enumerate(div.findall(T + "sp"), 1):
        speaker, *rest = sp
        rows.append({"act": div.get("n"), "n": n, "who": sp.get("who"), "speaker": speaker.text, "text": "".join(plain(e) for e in rest)})
speeches = pd.DataFrame(rows)
speeches.head(8)

## 3. 発言者を数え比べる

授業の教材で見た第1場の発言15・17と第2場の発言17を原資料から取り出します。次のセルで表を開いてください。

In [ ]:
#@title 操作：3発言の表を開く
sample = speeches[((speeches["act"] == "一") & speeches["n"].isin([15, 17])) | ((speeches["act"] == "二") & (speeches["n"] == 17))]
sample[["act", "n", "speaker", "who"]]

表示名ごとに数えると、3件は「小町」の一つのグループになります。作品内IDごとに数えると、グループはいくつになり、合計の件数は変わるでしょうか。表を読んで、`predicted_groups`にグループの数を、`total_changes`に`True`（変わる）か`False`（変わらない）を入れてから、次の集計を実行してください。

In [ ]:
predicted_groups = None
total_changes = None

In [ ]:
#@title 操作：予測した件数を確かめる
by_speaker = sample["speaker"].value_counts().to_dict()
by_who = sample["who"].value_counts().to_dict()
if predicted_groups is None or total_changes is None:
    print("予測：未記入です。上のセルに予測を入れてから、このセルをもう一度実行してください")
else:
    print("表示名別：", by_speaker)
    print("作品内ID別：", by_who)
    groups = "一致" if predicted_groups == len(by_who) else "不一致"
    total = "一致" if total_changes == (sum(by_who.values()) != sum(by_speaker.values())) else "不一致"
    print(f"予測：グループの数は{groups}、合計は{total}")

同じ数え方を戯曲全体に広げます。下のセルを実行し、`group_by`を`"who"`に変える前に、「小町」の件数の分かれ方と合計がどうなるか予測してください。変更後の合計を発言の行数と照合してください。

In [ ]:
group_by = "speaker"

counts = {}  # type: dict[str | int | None, int]
for speech in rows:
    key = speech[group_by]
    # 初めて出たキーの件数は0から始める
    counts[key] = counts.get(key, 0) + 1

print(counts)
print("合計発言数：", sum(counts.values()))
print("発言の行数：", len(rows))

### 一つの場を選ぶ

`act`は次の表に含める場を指定します。「一」で実行してから「二」に変え、「小町」の行の作品内IDと件数を確認してください。

In [ ]:
act = "一"
scene = speeches[speeches["act"] == act]
print("選んだ場の発言数：", len(scene))
scene.groupby(["who", "speaker"]).size().reset_index(name="count")

## 確認問題

第2場で、表示名が「小町」の発言は誰の発言ですか。3節の「一つの場を選ぶ」の表を読み、`answer`にその発言者のID（`who`の値）と発言の件数を入れてください。

In [ ]:
answer = {"who": "", "count": None}

In [ ]:
#@title 答えを確かめる
if not answer["who"] or answer["count"] is None:
    print("結果：不合格（`answer`に発言者のIDと件数を入れてください）")
elif answer == {"who": "#小野の小町", "count": 11}:
    print("結果：不合格（第1場の結果です。`act`を「二」に変えてください）")
elif answer["who"] != "#玉造の小町":
    print("結果：不合格（3節の「一つの場を選ぶ」の表で、`speaker`が「小町」の行の`who`を確かめてください）")
elif answer["count"] != 22:
    print("結果：不合格（同じ行の`count`を確かめてください）")
else:
    print("結果：合格")

### 授業外・任意：別の3発言で確かめる

次の表は第1場の発言14・15と第2場の発言17です。各人物が何回ずつ発言したかを調べるには、どの列を使いますか。

表を実行して読み、自分で列名と人物ごとの予想件数を手元に書いてください。予測を記録した後は、3節の例やAIによるエラーの説明を参照できます。

In [ ]:
#@title 操作：3発言の表を開く
trial = speeches[((speeches["act"] == "一") & speeches["n"].isin([14, 15])) | ((speeches["act"] == "二") & (speeches["n"] == 17))]
trial[["act", "n", "speaker", "who"]]

下の`"speaker"`を選んだ列名にして実行してください。予測と違ったら、同じキーに入った発言を表で探してください。列を変えても合計が3になる理由を一文で書いてください。

In [ ]:
group_by = "speaker"
counts = {}
for speech in trial.to_dict("records"):
    key = speech[group_by]
    counts[key] = counts.get(key, 0) + 1
print(counts)

### 授業外・任意：場と発言者の条件を書く

上の3行のうち、第2場で玉造の小町が話す行の発言番号を手元に書いてください。下のセルは戯曲全体から第2場の発言を数えます。

第2場で玉造の小町が話すときだけ1を足すように、`if`の行を書き換えてください。二つの条件は`and`でつなげます。結果を3節の第2場の表の件数と照合してください。

In [ ]:
count = 0
for speech in rows:
    if speech["act"] == "二":
        count += 1
print("条件に合う発言数：", count)

第2場で玉造の小町以外が話す発言はいくつありますか。第2場の総数と先ほどの結果から件数を予測し、発言者の条件を書き換えてください。二つの結果を足して、第2場の総数と照合してください。

二つの結果は同じ件数になるので、後の条件で選んだ発言者も確認してください。`count += 1`と同じ字下げで`print(speech["act"], speech["n"], speech["who"])`を加え、第2場の発言だけが並び、玉造の小町のIDが一度も出ないことを確かめてください。

### 授業外・任意：発言数と表記の出現回数を数え分ける

次のセルで第1場の発言19を読んでください。この一行だけを数えると、「あの人」を含む発言はいくつあり、「あの人」は何回出てきますか。次の集計を実行する前に、二つの数を手元に書いてください。

In [ ]:
print(speeches.loc[(speeches["act"] == "一") & (speeches["n"] == 19), "text"].iloc[0])

下のセルでは、同じ発言に「あの人」が二度あっても二つの件数に1ずつ足します。戯曲全体で表記が何回出たかを数えるには、`occurrences`に何を足せばよいですか。

`speech_text.count(phrase)`を使って一行を書き換えてください。二つの結果の差に発言19がどう関わるか、自分の予測と照合してください。予測と変更は自分で行い、実行時のエラーはAIに説明を求められます。

In [ ]:
phrase = "あの人"
matching_speeches = 0
occurrences = 0
for speech_text in speeches["text"]:
    if phrase in speech_text:
        matching_speeches += 1
        occurrences += 1
print("表記を含む発言数：", matching_speeches)
print("表記の出現回数：", occurrences)

## 4. 文字列検索で数える

ノートブックは、一つの検索語`word`を入力どおりの文字列として数えます。数える範囲で件数が変わります。

- **読みを含む本文**：ふりがなが名前の途中に入ると、検索に掛かりません。
- **読みを除いた本文**：発言者名の表示と舞台指示も数えます。
- **発言の中**：発言者名の表示と、発言の外の舞台指示を除きます。発言の途中の舞台指示は残ります。

`word`を「あの人」に変えると、誰を指すかは件数から分かりますか。

In [ ]:
word = "玉造の小町"

print("読みを含む本文：", "".join(body.itertext()).count(word))
print("読みを除いた本文：", plain(body).count(word))
# 正規表現で使う記号も、入力どおりの文字として数える
print("発言の中：", speeches["text"].str.count(re.escape(word)).sum())
speeches[speeches["text"].str.contains(word, regex=False)][["act", "n", "who", "text"]]

## 5. 注釈した言及を数える

文字列検索は、「あの人」が誰を指すかを区別しません。`material`は、第1場の発言14・15・19に人物のタグを付けた注釈案です。固有名には`persName`、代名詞などの参照表現には`rs type="person"`を付け、`ref="#tama"`で人物IDを指します。

発言19で「あの人」を囲むタグを二つ探してください。3発言のうち、固有名だけを数えると何件残るでしょうか。予測を記録してから、`include_references`を`False`に変えて表と照合してください。

In [ ]:
material = '''<body xmlns="http://www.tei-c.org/ns/1.0">
<sp xml:id="s1-sp14" who="#messenger"><speaker>使</speaker><p>いや、名前もあなたのように小町と云わなければいけないのです。</p></sp>
<sp xml:id="s1-sp15" who="#ono"><speaker>小町</speaker><p>小町！誰か小町と云う人はいなかったかしら。ああ、います。います。</p><stage>（発作的に笑い出しながら）</stage><p><persName ref="#tama">玉造の小町</persName>と云う人がいます。<rs type="person" ref="#tama">あの人</rs>を代りにつれて行って下さい。</p></sp>
<sp xml:id="s1-sp19" who="#ono"><speaker>小町</speaker><stage>（生き生きと）</stage><p>では<rs type="person" ref="#tama">あの人</rs>に行って貰って下さい。<rs type="person" ref="#tama">あの人</rs>はこの世にいるよりも、地獄に住みたいと云っています。誰も逢う人がいないものですから。</p></sp>
</body>'''

mention_word = "玉造の小町"
include_references = True

mentions = []
for e in ET.fromstring(material).iter():
    # rsはスイッチが有効で、人物を指すときだけ拾う
    if e.tag == T + "persName" or (include_references and e.tag == T + "rs" and e.get("type") == "person"):
        mentions.append({"text": "".join(e.itertext()), "id": e.get("ref", "")[1:]})
print("タグの付いた言及：", pd.DataFrame(mentions)["id"].value_counts().to_dict())
print(f"「{mention_word}」の文字列検索：", "".join(ET.fromstring(material).itertext()).count(mention_word))
pd.DataFrame(mentions)

### 授業外・任意：二つのIDを対応表で結ぶ

注釈の`tama`と原資料の`#玉造の小町`は、同じ人物を指す別々のIDです。`same_person`はその対応を宣言します。下のセルは対応をたどり、言及された人物が戯曲全体で何回発言したかを数えます。

3節の結果から発言数を予測してから実行してください。次に対応先を`"#小野の小町"`に変えると、発言数はいくつになるでしょうか。対応を一つ誤ると結果のどこが変わるかを、一文で書いてください。

In [ ]:
same_person = {"tama": "#玉造の小町"}

for mention_id, mention_count in pd.DataFrame(mentions)["id"].value_counts().items():
    who = same_person[mention_id]
    print(f"{mention_id}への言及：{mention_count}件、対応するID：{who}、その発言数：{(speeches['who'] == who).sum()}件")